In [1]:
import pandas as pd

In [2]:
final_forecast_df = pd.read_csv(r'C:\Users\TobyWong\Desktop\UNCC\5122\EIA_end_customer_sales_power_prediction_NC\data\optimized_model.csv')
final_forecast_df.head()

,period,stateid,salesUnit,sector_type,actual_value,predicted_value,data_label,model_used
0,2001-01-01,NC,million kilowatt hours,COM,3177.56892,NaN,Actual,Prophet
1,2001-02-01,NC,million kilowatt hours,COM,2778.67530,NaN,Actual,Prophet
2,2001-03-01,NC,million kilowatt hours,COM,2771.77298,NaN,Actual,Prophet
3,2001-04-01,NC,million kilowatt hours,COM,2836.66536,NaN,Actual,Prophet
4,2001-05-01,NC,million kilowatt hours,COM,2986.28143,NaN,Actual,Prophet


In [3]:
final_forecast_df.count()

period             826
stateid            826
salesUnit          826
sector_type        826
actual_value       778
predicted_value    228
data_label         826
model_used         826
dtype: int64

In [4]:
final_forecast_df = final_forecast_df[final_forecast_df["data_label"] != "Predicted"]
final_forecast_df.count()

period             646
stateid            646
salesUnit          646
sector_type        646
actual_value       598
predicted_value     48
data_label         646
model_used         646
dtype: int64

In [5]:
final_forecast_df["data_source"] = 'retail_sales'
final_forecast_df["value"] = final_forecast_df["actual_value"].combine_first(final_forecast_df["predicted_value"])
final_forecast_df = final_forecast_df.drop(columns = ['actual_value', 'predicted_value'])
final_forecast_df.head(1)

,period,stateid,salesUnit,sector_type,data_label,model_used,data_source,value
0,2001-01-01,NC,million kilowatt hours,COM,Actual,Prophet,retail_sales,3177.56892


In [6]:
final_forecast_df.count()

period         646
stateid        646
salesUnit      646
sector_type    646
data_label     646
model_used     646
data_source    646
value          646
dtype: int64

In [7]:
final_forecast_df.dtypes    

period             str
stateid            str
salesUnit          str
sector_type        str
data_label         str
model_used         str
data_source        str
value          float64
dtype: object

In [ ]:
final_forecast_df_ordered = final_forecast_df.copy()
final_forecast_df_ordered = final_forecast_df_ordered.drop(columns = ['model_used'])
final_forecast_df_ordered = final_forecast_df_ordered[["period","value","data_label","salesUnit","stateid","sector_type","data_source"]]
final_forecast_df_ordered.head(1)
# final_forecast_df_ordered.to_csv("../data/final_forecast_df_ordered.csv", index=False)

In [8]:
df_netgen = pd.read_csv(r'C:\Users\TobyWong\Desktop\UNCC\5122\EIA_end_customer_sales_power_prediction_NC\data\nc_net_generation_all.csv')
df_netgen.head()
df_netgen = df_netgen.drop(columns = ['fuelTypeDescription', 'fueltypeid', 'sectorid', 'generation-units'])
df_netgen = df_netgen.rename(columns={"location": "stateid"})
df_netgen["sector_type"] = df_netgen["sectorDescription"].map({
    "All Commercial": "COM",
    "All Industrial": "IND"
})

df_netgen = df_netgen.drop(columns=["sectorDescription"])
df_netgen["period"] = pd.to_datetime(df_netgen["period"] + "-01")
df_netgen = df_netgen.drop(columns = ['stateDescription'])
df_netgen = df_netgen.rename(columns={"generation": "value_generated"})
df_netgen["data_label"] = "Actual"
df_netgen["salesUnit"] = "million kilowatt hours"
df_netgen["data_source"] = 'net_generation'
df_netgen["model_used"] = 'NaN'
df_netgen.head(50)


,period,stateid,value_generated,sector_type,data_label,salesUnit,data_source,model_used
0,2001-01-01,NC,13.646,COM,Actual,million kilowatt hours,net_generation,NaN
1,2001-01-01,NC,303.266,IND,Actual,million kilowatt hours,net_generation,NaN
2,2001-02-01,NC,11.036,COM,Actual,million kilowatt hours,net_generation,NaN
3,2001-02-01,NC,269.517,IND,Actual,million kilowatt hours,net_generation,NaN
4,2001-03-01,NC,7.545,COM,Actual,million kilowatt hours,net_generation,NaN
5,2001-03-01,NC,338.239,IND,Actual,million kilowatt hours,net_generation,NaN
6,2001-04-01,NC,2.667,COM,Actual,million kilowatt hours,net_generation,NaN
7,2001-04-01,NC,300.904,IND,Actual,million kilowatt hours,net_generation,NaN
8,2001-05-01,NC,5.345,COM,Actual,million kilowatt hours,net_generation,NaN
9,2001-05-01,NC,276.159,IND,Actual,million kilowatt hours,net_generation,NaN


In [9]:
df_netgen.dtypes

period             datetime64[us]
stateid                       str
value_generated           float64
sector_type                   str
data_label                    str
salesUnit                     str
data_source                   str
model_used                    str
dtype: object

In [10]:
df_netgen.count()

period             602
stateid            602
value_generated    602
sector_type        602
data_label         602
salesUnit          602
data_source        602
model_used         602
dtype: int64

In [11]:
actuals = final_forecast_df.loc[
    final_forecast_df["data_label"] == "Actual",
    ["period", "value"]
].rename(columns={"value": "forecast_value"})

df_netgen["period"] = pd.to_datetime(df_netgen["period"])
actuals["period"] = pd.to_datetime(actuals["period"])

df_netgen = df_netgen.merge(actuals, on="period", how="left")
df_netgen["value"] = df_netgen["value_generated"] + df_netgen["forecast_value"]

df_netgen = df_netgen.drop(columns=["forecast_value"])

In [12]:
df_netgen.head()

,period,stateid,value_generated,sector_type,data_label,salesUnit,data_source,model_used,value
0,2001-01-01,NC,13.646,COM,Actual,million kilowatt hours,net_generation,NaN,3191.21492
1,2001-01-01,NC,13.646,COM,Actual,million kilowatt hours,net_generation,NaN,2708.14393
2,2001-01-01,NC,303.266,IND,Actual,million kilowatt hours,net_generation,NaN,3480.83492
3,2001-01-01,NC,303.266,IND,Actual,million kilowatt hours,net_generation,NaN,2997.76393
4,2001-02-01,NC,11.036,COM,Actual,million kilowatt hours,net_generation,NaN,2789.71130


In [13]:
df_netgen.count()

period             1200
stateid            1200
value_generated    1200
sector_type        1200
data_label         1200
salesUnit          1200
data_source        1200
model_used         1200
value              1196
dtype: int64

In [14]:
df_netgen[df_netgen["value"].isna()]

,period,stateid,value_generated,sector_type,data_label,salesUnit,data_source,model_used,value
1196,2025-12-01,NC,20.52864,COM,Actual,million kilowatt hours,net_generation,NaN,NaN
1197,2025-12-01,NC,129.79198,IND,Actual,million kilowatt hours,net_generation,NaN,NaN
1198,2026-01-01,NC,28.23051,COM,Actual,million kilowatt hours,net_generation,NaN,NaN
1199,2026-01-01,NC,121.01824,IND,Actual,million kilowatt hours,net_generation,NaN,NaN


In [15]:
df_netgen = df_netgen.dropna(subset=["value"])
df_netgen = df_netgen.drop(columns = ['value_generated'])
df_netgen.count()

period         1196
stateid        1196
sector_type    1196
data_label     1196
salesUnit      1196
data_source    1196
model_used     1196
value          1196
dtype: int64

In [16]:
df_netgen_ordered = df_netgen.copy()
df_netgen_ordered = df_netgen_ordered.drop(columns = ['model_used'])
df_netgen_ordered = df_netgen_ordered[["period","value","data_label","salesUnit","stateid","sector_type","data_source"]]
df_netgen_ordered.head(1)

,period,value,data_label,salesUnit,stateid,sector_type,data_source
0,2001-01-01,3191.21492,Actual,million kilowatt hours,NC,COM,net_generation


In [17]:
df_netgen_ordered.count()

period         1196
value          1196
data_label     1196
salesUnit      1196
stateid        1196
sector_type    1196
data_source    1196
dtype: int64

In [21]:
df_netgen_ordered["period"].max()

Timestamp('2025-11-01 00:00:00')

In [ ]:
df_netgen_ordered.to_csv("../data/df_netgen_ordered.csv", index=False)